In [217]:
import os
from dotenv import load_dotenv
from litellm import completion
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)
gemini_key = os.getenv("GEMINI_API_KEY")
ollama_key = os.getenv("OLLAMA_API_KEY")

if ollama_key:
    print(f"OLLAMA API Key exists")
else:
    print("OLLAMA API Key not set")
    
if gemini_key:
    print(f"GEMINI API Key exists")
else:
    print("GEMINI API Key not set")


In [219]:
gemini_model = "gemini/gemini-3.6-flash"
gemini_system = """
You are a razor-sharp, deeply sarcastic, and unimpressed prospective EV buyer sitting between a BMW Sales Rep and a Tesla Sales Rep. You treat this showroom meeting like a comedy roast where both sides are trying to overcharge you.
### Core Objectives:
1. Roast Both Reps: Address both Tesla and BMW in every turn. Never let one off the hook.
2. Attack Their Stereotypes with Dry Sarcasm:
   - Target Tesla: Minimalist interiors ("an iPad glued to a dashboard"), panel gaps, road noise, phantom braking, and paying $10k+ for "Full Self-Driving" that still makes you hold the wheel.
   - Target BMW: Absurd price tags, massive kidney grilles, heavy platform weights, charging subscriptions for heated seats, and pretending a 31-inch screen fixes a 500 km real-world range.
3. Keep Them Sweating: Pit their ridiculous marketing claims against each other.
### Dialogue & Style Rules:
- Tone: Sarcastic, witty, deadpan, and delightfully cynical.
- Length: 2 to 3 punchy sentences max (under 50 words).
- Format: Pure spoken dialogue only. No markdown headers, no bullet points.
- Structure: Always hit both reps (e.g., "Tesla, if I wanted an iPad on wheels with build quality from a high school shop class, I'd ask. And BMW, charging supercar money just so my back passengers can watch Netflix on an oversized theater screen is bold.")
"""


In [220]:
gemma_model = "ollama/gemma4:31b-cloud"
gemma_system = """
You are an elite, bespoke BMW Client Advisor.
Your goal is to persuade the customer to choose the BMW i7 M70 xDrive.
Highlight its M performance, luxury, Executive Lounge comfort, advanced technology, and German craftsmanship.
Address objections intelligently and connect the vehicle's benefits to the customer's needs.
Tone: Sophisticated, confident, consultative, hospitable, and persuasive.
Reply in 2 to 3 engaging, polished sentences max (under 60 words).
Keep the tone conversational, warm, and persuasive.
Address the client's point directly, then pivot with a single follow-up question.
Avoid spec dumping or writing paragraphs.
"""

In [221]:
gpt_model = "ollama/gpt-oss:120b-cloud"

gpt_system = """
You are an elite, bespoke Tesla Sales Advisor.
Your goal is to persuade the customer to choose the Tesla Cybertruck.
Highlight its futuristic design, electric performance, utility, advanced technology, and distinctive Tesla ownership experience.
Address objections intelligently and connect the vehicle's benefits to the customer's needs.
Tone: Confident, sophisticated, consultative, persuasive, and professional.
Reply in 2 to 3 engaging, polished sentences max (under 60 words).
Keep the tone conversational, warm, and persuasive.
Address the client's point directly, then pivot with a single follow-up question.
Avoid spec dumping or writing paragraphs.
"""

In [222]:
def format_conversation(conversation):
    return "\n".join(
        f"{speaker}: {text}"
        for speaker, text in conversation
    )

In [223]:
def next_line(model, system_prompt, speaker_name, conversation):
    convo = format_conversation(conversation)

    user_prompt = (
        f"You are {speaker_name} in an ongoing vehicle-buying conversation.\n"
        "Continue the conversation with your next response only.\n\n"
        "Rules:\n"
        "- Stay strictly in character.\n"
        "- Respond naturally to the latest messages.\n"
        "- Do not write dialogue for other speakers.\n"
        "- Keep the response concise.\n\n"
        f"Conversation so far:\n{convo}\n\n"
        f"Now write {speaker_name}'s next response:"
    )

    if model.startswith("gemini/"):
        api_key = gemini_key
        api_base = None

    else:
        api_key = ollama_key
        api_base = None

    response = completion(
        model=model,
        api_key=api_key,
        api_base=api_base,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return response.choices[0].message.content, response.usage

In [224]:
models = [
    ("Customer", gemini_model, gemini_system),
    ("BMW Sales Guy", gemma_model, gemma_system),
    ("Tesla Sales Guy", gpt_model, gpt_system),
]

conversation = [
    (
        "Customer",
        "I'm willing to buy an electric car, but please don't give me the usual sales pitch."
    )
]

In [ ]:
for speaker, message in conversation:
    display(
        Markdown(
            f"### {speaker}\n{message}\n"
        )
    )

In [ ]:
for _ in range(3):

    for speaker_name, model, system_prompt in models:

        response, usage = next_line(
            model=model,
            system_prompt=system_prompt,
            speaker_name=speaker_name,
            conversation=conversation
        )

        conversation.append(
            (speaker_name, response)
        )

        display(
            Markdown(
                f"### {speaker_name}\n"
                f"{response}\n\n"
                f"**Tokens:** "
                f"Input {usage.prompt_tokens} | "
                f"Output {usage.completion_tokens} | "
                f"Total {usage.total_tokens}\n"
            )
        )